In [ ]:
import av
from matplotlib import pyplot as plt
import numpy as np
import cv2
from glob import glob
import os

In [ ]:
def get_lines(crop):
    lines = []

    for idx, line in enumerate(crop):
        if len(line) > 0:
            lines.append((
                (line[2], line[3]),
                (line[0], line[1]),
                (255, 0, 0) if idx % 2 == 0 else (0, 255, 0)
            ))

    return lines

def draw_crop(frame, crop):
    for vertical_crop, horizontal_crop, color in get_lines(crop):
        frame_pos = (horizontal_crop[0], vertical_crop[0])
        frame_size = (
            frame.shape[1] - horizontal_crop[1],
            frame.shape[0] - vertical_crop[1],
        )

        frame = cv2.rectangle(frame, frame_pos, frame_size, color, 5)

    return frame

def create_crop(frame, crop):
    combined_frame = None
    lines = get_lines(crop)

    for vertical_crop, horizontal_crop, _ in lines:
        cropped_line = frame[
            vertical_crop[0]:-vertical_crop[1],
            horizontal_crop[0]:-horizontal_crop[1],
        ]

        if combined_frame is None:
            combined_frame = cropped_line
        else:
            if combined_frame.shape[0] < cropped_line.shape[0]:
                combined_frame = np.pad(
                    combined_frame,
                    ((0, cropped_line.shape[0] - combined_frame.shape[0]), (0, 0), (0, 0)),
                    mode='constant',
                    constant_values=0
                )
            elif combined_frame.shape[0] > cropped_line.shape[0]:
                cropped_line = np.pad(
                    cropped_line,
                    ((0, combined_frame.shape[0] - cropped_line.shape[0]), (0, 0), (0, 0)),
                    mode='constant',
                    constant_values=0
                )

            combined_frame = np.concatenate((combined_frame, cropped_line), axis=1)

    return combined_frame

In [ ]:
crop_dict = {
    #                 left line (front)  ,   right line (back)
    #               (l  , r   , t  , b  ),
    "JS-BM-P487":  ((450, 850 , 650, 20 ), (1100, 20 , 650, 20 )),
    "JK-CE-P571":  ((                   ), (20  , 20 , 150, 20 )),
    "JS-BM-P488":  ((20 , 900 , 220, 130), (1100, 20 , 220, 130)),
    "JS-BM-P489":  ((20 , 750 , 300, 130), (1220, 20 , 300, 130)),
    "JS-BM-P490":  ((20 , 970 , 380, 20 ), (1030, 150, 380, 20 )),
    "JS-BM-P491":  ((20 , 1100, 250, 20 ), (880 , 250, 250, 20 )),
    "JS-BM-P493":  ((20 , 890 , 230, 20 ), (1050, 150, 200, 20 )),
    "JS-BM-P4951": ((20 , 390 , 120, 20 ), (350 , 20 , 120, 20 )),
    "JS-BM-P498":  ((20 , 800 , 300, 130), (1190, 20 , 300, 130)),
    "JS-CM-P492":  ((20 , 950 , 250, 130), (1090, 20 , 250, 130)),
    "JU-PO-P573":  ((20 , 340 , 150, 20 ), (415 , 60 , 150, 20 )),
    "PJ-CE-P551":  ((20 , 370 , 300, 20 ), (400 , 20 , 300, 20 )),
    "SD-PO-P576":  ((20 , 335 , 200, 20 ), (405 , 20 , 200, 20 )),
    "SG-CE-P574":  ((20 , 385 , 135, 20 ), (365 , 20 , 135, 20 )),
    "SU-CE-P575":  ((20 , 385 , 150, 20 ), (360 , 20 , 150, 20 )),
}

In [ ]:
root_path = "/home/simon/SASTRASAT/videos/20250519_sample/"

# for all video files in the root path
for video_file in sorted(glob("*.mp4", root_dir=root_path)):
    full_video_path = os.path.join(root_path, video_file)

    try:
        container = av.open(full_video_path)
    except:
        print(f"Error opening video file: {full_video_path}")
        continue

    frame_raw = next(container.decode(video=0))

    frame = np.array(frame_raw.to_image())

    crop = (20, 20, 20, 20), (20, 20, 20, 20)
    for video_name, video_crop in crop_dict.items():
        if video_file.startswith(video_name):
            crop = video_crop
            break

    frame = create_crop(frame, crop)
    print(video_file)
    plt.imshow(frame)
    plt.show()